# Experiment 3 — Data Pre-processing

**CSE4007L — Applications of Machine Learning in Industry**  
Com-Project 2: Fake News Detection

Krishna Kaushik · 230160223030 · Section-A Group 2

**Before running:** put `Fake.csv` and `True.csv` in the same folder as this notebook
(or set `PATH` in the setup cell). Then **Run All**.

    pip3 install pandas scikit-learn joblib

Produces `train.csv`, `val.csv`, `test.csv` and `tfidf.joblib` — Experiment 4 needs `train.csv`.


In [1]:
import re, joblib, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

PATH = ""          # folder containing Fake.csv and True.csv ("" = same folder)

## 1. Load the raw dataset

In [2]:
fake = pd.read_csv(PATH + "Fake.csv");  fake["label"] = 1     # 1 = fake
true = pd.read_csv(PATH + "True.csv");  true["label"] = 0     # 0 = real
df = pd.concat([true, fake], ignore_index=True)
print("Fake:", fake.shape, " True:", true.shape, " Combined:", df.shape)
print(df.head(3))

Fake: (23481, 5)  True: (21417, 5)  Combined: (44898, 5)
                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept transgender recruits o...   
2  Senior U.S. Republican senator: 'Let Mr. Muell...   

                                                text       subject  \
0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1  WASHINGTON (Reuters) - Transgender people will...  politicsNews   
2  WASHINGTON (Reuters) - The special counsel inv...  politicsNews   

                 date  label  
0  December 31, 2017       0  
1  December 29, 2017       0  
2  December 31, 2017       0  


## 2. Handle missing values

In [3]:
print("Total nulls      :", df.isnull().sum().sum())
print("Empty articles   :", (df["text"].str.strip() == "").sum())
print("Duplicate rows   :", df.duplicated(subset=["title", "text"]).sum())

df = df[df["text"].str.strip() != ""]
df = df.drop_duplicates(subset=["title", "text"]).reset_index(drop=True)
print("After cleaning   :", df.shape)

Total nulls      : 0
Empty articles   : 631
Duplicate rows   : 5793
After cleaning   : (38658, 5)


## 3. Separate feature X and target y

In [4]:
def clean(t):
    t = t.lower()
    t = re.sub(r"reuters|http\S+|<.*?>", " ", t)   # publisher name, URLs, HTML
    t = re.sub(r"[^a-z\s]", " ", t)                # keep letters only
    return " ".join(t.split())

X = (df["title"] + " " + df["text"]).apply(clean)         # feature
y = df["label"]                                           # target
print("X:", X.shape, " y:", y.shape)
print(y.value_counts(normalize=True).round(4))

X: (38658,)  y: (38658,)
label
0    0.5483
1    0.4517
Name: proportion, dtype: float64


## 4. Train / validation / test split

In [5]:
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)
print("Train / Val / Test:", len(X_tr), "/", len(X_va), "/", len(X_te))

tfidf = TfidfVectorizer(max_features=50000, stop_words="english")
X_tr_v = tfidf.fit_transform(X_tr)      # fit on train only
X_va_v = tfidf.transform(X_va)
X_te_v = tfidf.transform(X_te)
print("TF-IDF train matrix:", X_tr_v.shape)

Train / Val / Test: 27060 / 5799 / 5799
TF-IDF train matrix: (27060, 50000)


## 5. Save the clean dataset

In [6]:
pd.DataFrame({"text": X_tr, "label": y_tr}).to_csv("train.csv", index=False)
pd.DataFrame({"text": X_va, "label": y_va}).to_csv("val.csv", index=False)
pd.DataFrame({"text": X_te, "label": y_te}).to_csv("test.csv", index=False)
joblib.dump(tfidf, "tfidf.joblib")
print("Saved: train.csv, val.csv, test.csv, tfidf.joblib")

Saved: train.csv, val.csv, test.csv, tfidf.joblib


## 6. Summary

In [7]:
print("Clean records  :", len(df))
print("Real / Fake    :", (y == 0).sum(), "/", (y == 1).sum())
print("Split          :", len(X_tr), "/", len(X_va), "/", len(X_te))
print("TF-IDF features:", X_tr_v.shape[1])

Clean records  : 38658
Real / Fake    : 21196 / 17462
Split          : 27060 / 5799 / 5799
TF-IDF features: 50000
